# Holographic Lattice Memory

Mirrors `08_holographic_memory.ipynb`.  The temporal pyramid accumulates
quantized V-JEPA tid streams into layers; the top HLLSet plus the TF stack
reconstructs any past state (the holographic property, `main.tex` §Noether).

In [ ]:
import sys
import pathlib

ROOT = pathlib.Path.cwd()
if not (ROOT / "ewm_jepa").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import torch
from ewm_jepa import Quantizer
import hllset_py
from hllset_cortex import HLLSetFilter
from hllset_cortex.domain import encoding_tokenizer, hllset_from_ids, tid

# A deterministic tid-stream source: the same quantizer the JEPA pipeline
# uses, fed with synthetic encodings.  The EWM boundary is encoding-agnostic —
# these tids are interchangeable with V-JEPA quantized patch encodings.
def make_tid_stream(n=200, dim=1024, codebook_size=2048, seed=0):
    q = Quantizer(dim=dim, codebook_size=codebook_size, device="cpu", seed=seed)
    # decorrelate the data seed from the codebook seed so streams differ
    z = torch.randn(n, dim, generator=torch.Generator().manual_seed(1000 + seed))
    return q.ids_to_text(q.encode(z))


## 1. The temporal pyramid over tid streams

In [ ]:
from hllset_cortex.temporal import drn

class Pyramid:
    """Configurable sliding-window pyramid over real HLLSets (the §4.2
    construction, with the top-layer carry guarded)."""
    def __init__(self, durations):
        self.durations = list(durations)
        self.layers = [hllset_py.HLLSet() for _ in range(len(durations) + 1)]
        self.counters = [0] * len(durations)

    @property
    def top(self):
        top = hllset_py.HLLSet()
        for layer in self.layers:
            top = top.union(layer)
        return top

    def step(self, s):
        self.layers[0] = self.layers[0].union(s)
        self.counters[0] += 1
        for i, d in enumerate(self.durations):
            if self.counters[i] < d:
                break
            self.layers[i + 1] = self.layers[i + 1].union(self.layers[i])
            self.layers[i] = hllset_py.HLLSet()
            self.counters[i] = 0
            if i + 1 < len(self.counters):
                self.counters[i + 1] += 1

    def layer_popcounts(self):
        return [layer.popcount() for layer in self.layers]

pyramid = Pyramid([3, 2, 2])   # 4 layers, compressed config
print("layers:", len(pyramid.layers))

for step in range(12):
    s = make_tid_stream(n=30, seed=step)
    pyramid.step(hllset_py.HLLSet.from_tokens(s.split()))
    if step in (0, 2, 3, 5, 11):
        print(f"step {step:2d}: layers={pyramid.layer_popcounts()} "
              f"top={pyramid.top.popcount()}")


## 2. DRN decomposition — the Noether balance

Each arriving observation splits into Departed / Retained / Novel against the
previous system state.  The steering condition `|card(N) - card(D)| -> 0`
is the EWM analog of homeostatic reinforcement balance.

In [ ]:
prev = hllset_py.HLLSet()
for step in range(8):
    s = hllset_py.HLLSet.from_tokens(make_tid_stream(n=40, seed=step).split())
    d = drn(s, prev)
    print(f"step {step}: R={d.r.popcount()} D={d.d.popcount()} N={d.n.popcount()} "
          f"| |N|-|D|| = {abs(d.n.popcount() - d.d.popcount())}")
    prev = prev.union(s)


## 3. Holographic reconstruction

The top HLLSet never loses bits (union invariant), and the LUT records *when*
each tid was active.  Past states are recovered by intersecting the top with
the TF-ranked tid set of a past window — the TF stack is the reference beam.

In [ ]:
f = HLLSetFilter()
f.tokenizer = encoding_tokenizer()
states = []
for step in range(6):
    s = make_tid_stream(n=40, seed=step)
    f.process_text(s)
    states.append(hllset_py.HLLSet.from_tokens(s.split()))

top = hllset_py.HLLSet()
for s in states:
    top = top.union(s)

def reconstruct(past):
    return top.intersection(past)

for step in (0, 3, 5):
    r = reconstruct(states[step])
    print(f"reconstruct step {step}: {r.popcount()} / {states[step].popcount()} bits "
          f"({r.popcount() / max(states[step].popcount(), 1):.3f})")


## 4. Recovery ratio over time

Older states remain reconstructible because the lattice never forgets:
exclusion is attention, not erasure.

In [ ]:
ratios = []
for step, s in enumerate(states):
    ratios.append(reconstruct(s).popcount() / max(s.popcount(), 1))
print("recovery ratios:", [round(r, 3) for r in ratios])


## 5. Summary

The holographic property is a **Noether consequence**: because the system
state is a monotonic union, information is never lost from the top — it is
only redistributed across layers.  The same argument applies when the
streams are quantized V-JEPA encodings of a video corpus: the lattice becomes
a holographic memory of what the world model has perceived.